# CS 506 Review

A review of the main ideas behind correlation, distances, and clustering, focused on *why* each tool behaves the way it does. Each section has a handful of exercises: some ask you to implement a metric or algorithm, some to predict what will happen before running a cell, and some to explain what you see. Answer in the empty cells and fill in the code wherever you see `...` or `# YOUR CODE HERE`.

Where a worksheet already has you implement an algorithm, the section starts with a note to complete that worksheet first, and the exercises here use the scikit-learn version instead.

| Section | Topic |
|---|---|
| 1 | Correlation |
| 2 | Distance & similarity |
| 3 | K-means and its relatives |
| 4 | Hierarchical clustering |
| 5 | DBSCAN |
| 6 | Gaussian mixture models and EM |
| 7 | Clustering aggregation |
| 8 | Which tool for which data? |

Run the setup cell first. Each section only depends on the setup cell, so you can do them in any order.

## Setup

Run this cell first. Every dataset is either generated in code or bundled with scikit-learn, so no files need to be downloaded.

In [ ]:
# If you are running locally and something is missing:
# pip install numpy pandas matplotlib scipy scikit-learn
import time
import itertools
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["figure.dpi"] = 90
plt.rcParams["axes.grid"] = False
print("Setup complete")

## 1. Correlation: how similar are two features?

Given $n$ data points and two features (columns) $x=(x_1,\dots,x_n)$ and $y=(y_1,\dots,y_n)$, Pearson correlation $r(x,y)$ is a score for their linear association; this section builds it from scratch and then probes what it does and does not capture.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_similarity

### 1.1 Correlation and cosine similarity, from their formulas

$$r(x,y) = \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sqrt{\sum_i (x_i - \bar x)^2}\,\sqrt{\sum_i (y_i - \bar y)^2}}$$

Viewing $x$ and $y$ as vectors in $\mathbb R^n$, this is exactly the cosine of the angle between their **mean-centered vectors** $\tilde x = x-\bar x$ and $\tilde y = y-\bar y$, where the cosine similarity of any two vectors $u,v$ is

$$\cos(u,v) = \frac{u\cdot v}{\lVert u\rVert\,\lVert v\rVert}$$

**a)** (i) Implement `correlation` and `cosine` directly from the two formulas above, using only `np.sum`/`np.dot`/`np.sqrt`/`np.linalg.norm` (no `np.corrcoef`, `np.cov`, `np.var`). Check that $r(x,y)$ equals the cosine of the centered vectors. (ii) Height in cm and shoe price in dollars are generated independently below, so $r\approx0$. Before running the plot cell, predict whether the *plain* (uncentered) cosine of these two vectors is near $0$, and how the plain cosine and $r$ change as we add a constant offset $c\in[-300,300]$ to the prices. Explain why centering matters.

In [ ]:
def correlation(x, y):
    """Pearson correlation r(x, y): dot of deviation vectors over the product of their norms."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    # YOUR CODE HERE
    ...


def cosine(u, v):
    """Cosine of the angle between vectors u and v: u.v / (||u|| ||v||)."""
    u, v = np.asarray(u, dtype=float), np.asarray(v, dtype=float)
    # YOUR CODE HERE
    ...


rng = np.random.default_rng(0)
x = rng.normal(size=50)
y = 0.6 * x + rng.normal(size=50)

assert np.isclose(correlation(x, y), np.corrcoef(x, y)[0, 1])
assert np.isclose(cosine(x, y), sk_cosine_similarity(x[None, :], y[None, :])[0, 0])
assert np.isclose(correlation(x, y), cosine(x - x.mean(), y - y.mean()))
print(f"our r = {correlation(x, y):.4f}, np.corrcoef = {np.corrcoef(x, y)[0, 1]:.4f}, "
      f"cosine of centered vectors = {cosine(x - x.mean(), y - y.mean()):.4f}")

In [ ]:
rng = np.random.default_rng(11)
height_cm = rng.normal(170, 8, size=100)
shoe_price = rng.normal(80, 15, size=100)               # generated independently of height
print(f"r(height, price) = {correlation(height_cm, shoe_price):+.3f}    cosine(height, price) = {cosine(height_cm, shoe_price):+.3f}")

offsets = np.linspace(-300, 300, 241)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(offsets, [cosine(height_cm, shoe_price + c) for c in offsets], lw=2, label="plain cosine(height, price + c)")
ax.plot(offsets, [correlation(height_cm, shoe_price + c) for c in offsets], lw=2, label="r(height, price + c)")
ax.axhline(0, color="0.7", lw=0.8)
ax.set_xlabel("offset c added to price ($)"); ax.set_ylabel("similarity"); ax.set_ylim(-1.05, 1.05)
ax.set_title("Adding a constant moves the cosine, not r", fontsize=10)
ax.legend(loc="lower right", fontsize=8); plt.tight_layout(); plt.show()

*Your answer here.*

### 1.2 The power of a single point

Take a clean $y\approx x$ cloud and add **one** extra point at $(p_x,p_y)$. The heatmaps show the resulting $r$ for every position of that point (solid contour: $r=0$; dashed: $r$ unchanged), for a cloud of $n=10$ and one of $n=200$; the right panel moves the extra point away from the centroid in four fixed directions.

**b)** Before running, sketch where the extra point must go to make $r\approx0$ and to make $r$ negative. Then use the figure to answer: (i) which directions of movement change $r$ fastest, and which barely matter? Explain using what the extra point adds to the numerator and the two sums of squares in $r$. (ii) Why is the $n=10$ correlation so much more fragile than the $n=200$ one, and roughly how does the distance needed to cancel $r$ scale with $n$?

In [ ]:
def influence_map(x, y, grid):
    """R[i, j] = correlation of the data after appending ONE extra point at (grid[j], grid[i])."""
    R = np.empty((len(grid), len(grid)))
    for i, py in enumerate(grid):
        for j, px in enumerate(grid):
            R[i, j] = correlation(np.append(x, px), np.append(y, py))
    return R


grid = np.linspace(-25, 35, 121)
directions = {"along (1,1)": (1, 1), "across (1,-1)": (1, -1), "horizontal (1,0)": (1, 0), "vertical (0,1)": (0, 1)}
ts = np.linspace(0, 60, 121)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), layout="constrained")
for ax, n_pts, ls in [(axes[0], 10, "-"), (axes[1], 200, "--")]:
    rng = np.random.default_rng(6)
    xs = rng.uniform(0, 10, n_pts)
    ys = xs + rng.normal(0, 1.0, n_pts)
    R, r0 = influence_map(xs, ys, grid), correlation(xs, ys)
    im = ax.pcolormesh(grid, grid, R, cmap="RdBu_r", vmin=-1, vmax=1, shading="auto")
    if R.min() < 0:
        ax.contour(grid, grid, R, levels=[0], colors="k", linewidths=1.2)
    ax.contour(grid, grid, R, levels=[r0], colors="k", linewidths=1.2, linestyles="--")
    ax.scatter(xs, ys, s=6, c="k"); ax.set_aspect("equal")
    ax.set_title(f"n={n_pts}: base r={r0:.2f}, min r={R.min():.2f}\nshare of grid with r<0: {(R < 0).mean():.0%}", fontsize=9)
    ax.set_xlabel("x of extra point"); ax.set_ylabel("y of extra point")
    cx, cy = xs.mean(), ys.mean()
    for k, (name, (dx, dy)) in enumerate(directions.items()):
        d = np.array([dx, dy]) / np.hypot(dx, dy)
        axes[2].plot(ts, [correlation(np.append(xs, cx + t * d[0]), np.append(ys, cy + t * d[1])) for t in ts],
                     color=f"C{k}", ls=ls, lw=2, label=f"{name}, n={n_pts}")
fig.colorbar(im, ax=axes[1], label="r (heatmaps)", shrink=0.85)
axes[2].axhline(0, color="0.8", lw=0.8); axes[2].set_ylim(-1.05, 1.05)
axes[2].set_xlabel("distance of extra point from origin"); axes[2].set_ylabel("r")
axes[2].set_title("Moving the extra point (solid n=10, dashed n=200)", fontsize=9); axes[2].legend(fontsize=7, ncol=2)
plt.show()

*Your answer here.*

### 1.3 Zero correlation does not mean "no relationship"

**c)** For each *perfectly deterministic* relationship below ($x$ evenly spaced), predict $r$: about $0$, positive, negative, or exactly $\pm1$? (1) $y=x^2$ on $[-3,3]$; (2) points on a circle; (3) $y=|x|$ on $[-3,3]$; (4) $y=\cos x$ on $[0,2\pi]$; (5) $y=\sin x$ on $[0,2\pi]$; (6) $y=e^x$ on $[0,5]$.

In [ ]:
xs_sym = np.linspace(-3, 3, 61)
x_per = np.linspace(0, 2 * np.pi, 121)
theta = np.linspace(0, 2 * np.pi, 120, endpoint=False)
x_exp = np.linspace(0, 5, 61)
shapes = {
    "y = x²":              (xs_sym, xs_sym ** 2),
    "circle":              (np.cos(theta), np.sin(theta)),
    "y = |x|":             (xs_sym, np.abs(xs_sym)),
    "y = cos x on [0,2π]": (x_per, np.cos(x_per)),
    "y = sin x on [0,2π]": (x_per, np.sin(x_per)),
    "y = eˣ on [0,5]":     (x_exp, np.exp(x_exp)),
}
fig, axes = plt.subplots(2, 3, figsize=(10, 5.2))
for ax, (name, (xv, yv)) in zip(axes.ravel(), shapes.items()):
    ax.scatter(xv, yv, s=6)
    ax.axvline(xv.mean(), color="0.8", lw=0.8); ax.axhline(yv.mean(), color="0.8", lw=0.8)
    ax.set_title(f"{name}:  r = {np.corrcoef(xv, yv)[0, 1]:+.3f}", fontsize=9)
axes[0, 1].set_aspect("equal")
plt.tight_layout(); plt.show()
print(f"exp on [0, 5]:  r(x, e^x) = {np.corrcoef(x_exp, np.exp(x_exp))[0, 1]:.4f}   but   r(x, log(e^x)) = {np.corrcoef(x_exp, np.log(np.exp(x_exp)))[0, 1]:.4f}")

*Your answer here.*

## 2. Distance & similarity: how similar are two data points?

Every clustering algorithm later in the course is only as good as its answer to "how far apart are two points?". Here you build the core dissimilarities and use **nearest-neighbour retrieval** (does the most similar stored point have the right identity or label?) to see that the data should decide which one to use.

In [ ]:
from collections import Counter
from scipy.spatial import distance as ssd
from scipy.spatial.distance import pdist, cdist
from sklearn.metrics import pairwise_distances
from sklearn.metrics.pairwise import cosine_similarity

### 2.1 The toolbox

- **Minkowski:** $d_p(x,y) = \big(\sum_{i=1}^d |x_i - y_i|^p\big)^{1/p}$; $p=1$ is Manhattan, $p=2$ Euclidean, and the limit $p\to\infty$ is Chebyshev, $d_\infty(x,y)=\max_i |x_i-y_i|$.
- **Cosine:** $s(x,y) = \dfrac{x\cdot y}{\|x\|_2\,\|y\|_2}$, with dissimilarity $1 - s(x,y)$.
- **Jaccard:** $J(A,B) = \dfrac{|A\cap B|}{|A\cup B|}$ for sets, distance $1 - J(A,B)$. For binary vectors, $u_i = 1$ means "item $i$ is in the set".

**a)** Complete the functions below. `minkowski` must handle any $p > 0$ and `p = np.inf` (use a `max`; a huge exponent overflows). Treat two empty sets as identical (Jaccard distance $0$). Then run the check cell.

In [ ]:
def minkowski(x, y, p):
    """L_p distance between 1-D arrays x and y; p > 0 or np.inf (Chebyshev)."""
    # YOUR CODE HERE
    ...

def cosine_sim(x, y):
    """x·y / (||x|| ||y||); assumes neither vector is all zeros."""
    # YOUR CODE HERE
    ...

def cosine_dist(x, y):
    return ...

def jaccard_dist_sets(A, B):
    """1 - |A ∩ B| / |A ∪ B| for two Python sets (0.0 if both are empty)."""
    # YOUR CODE HERE
    ...

def jaccard_dist_binary(u, v):
    """Jaccard distance for two 0/1 vectors of equal length (0.0 if both are all zeros)."""
    # YOUR CODE HERE
    ...

In [ ]:
# Check your work against scipy
rng = np.random.default_rng(0)
for _ in range(200):
    x, y = rng.normal(size=(2, 7))
    for p in [1, 1.5, 2, 3]:
        assert np.isclose(minkowski(x, y, p), ssd.minkowski(x, y, p)), p
    assert np.isclose(minkowski(x, y, np.inf), ssd.chebyshev(x, y))
    assert np.isclose(cosine_dist(x, y), ssd.cosine(x, y))
    u, v = rng.random((2, 12)) < 0.4
    if (u | v).any():
        assert np.isclose(jaccard_dist_binary(u, v), ssd.jaccard(u, v))
        assert np.isclose(jaccard_dist_sets(set(np.flatnonzero(u)), set(np.flatnonzero(v))), ssd.jaccard(u, v))
assert jaccard_dist_sets(set(), set()) == 0 and np.isclose(cosine_sim([1, 2], [3, 6]), 1)
print("minkowski, cosine and jaccard match scipy")

*Your answer here.*

### 2.2 The data decides: Manhattan, Euclidean or Chebyshev?

A database holds $N = 300$ items with $d = 20$ features. A query is a **corrupted copy** of one stored item; retrieval succeeds if the query's $L_p$ nearest neighbour is the original. Three noise models:
**glitch** — one random feature is off by a huge $\pm G$ (a sensor spike), the rest are nearly exact;
**Gaussian** — every feature gets independent $\mathcal N(0,\sigma^2)$ noise;
**bounded** — every feature is off by up to $\pm a$, uniformly (a tolerance).

**b)** Before running the next cell, rank $p \in \{1, 2, \infty\}$ from best to worst retrieval for each noise model.

*Your answer here.*

In [ ]:
def lp_matrix(Q, X, p):
    """L_p distances between the rows of Q (n x d) and the rows of X (m x d)."""
    diff = np.abs(Q[:, None, :] - X[None, :, :])
    return diff.max(-1) if np.isinf(p) else (diff ** p).sum(-1) ** (1 / p)

def corrupt(X_true, model, level, rng):
    """Noisy copy of the rows of X_true under the given noise model."""
    Q = X_true.copy()
    if model == "glitch":
        Q += rng.normal(0, 0.1, Q.shape)                      # tiny noise everywhere
        j = rng.integers(Q.shape[1], size=len(Q))             # one glitched feature per query
        Q[np.arange(len(Q)), j] += ...   # +G or -G
    elif model == "gaussian":
        Q += rng.normal(0, level, Q.shape)
    elif model == "bounded":
        Q += rng.uniform(-level, level, Q.shape)
    return Q

rng = np.random.default_rng(8)
N_db, d_db, T = 300, 20, 300
X_items = rng.normal(size=(N_db, d_db))
ps_ret = [1, 2, np.inf]
levels = {"glitch": [0, 2, 4, 6, 8, 12, 16, 24, 32],
          "gaussian": [0.25, 0.5, 0.75, 1.0, 1.25, 1.5],
          "bounded": [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]}
acc = {m: {p: [] for p in ps_ret} for m in levels}
for model, lvls in levels.items():
    for lvl in lvls:
        true_idx = rng.integers(N_db, size=T)
        Q = corrupt(X_items[true_idx], model, lvl, rng)
        for p in ps_ret:
            acc[model][p].append(...)   # fraction retrieved correctly

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
xlabels = {"glitch": "spike size $G$", "gaussian": r"noise std $\sigma$", "bounded": "tolerance $a$"}
for ax, model in zip(axes, levels):
    for p in ps_ret:
        ax.plot(levels[model], acc[model][p], "o-", ms=4, label=f"p = {p}")
    ax.set_title(f"{model} noise"); ax.set_xlabel(xlabels[model])
axes[0].set_ylabel("retrieval accuracy"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
for model, i in [("glitch", 4), ("gaussian", 3), ("bounded", 3)]:
    print(f"{model:9s} at level {levels[model][i]}: " + ", ".join(f"p={p}: {acc[model][p][i]:.2f}" for p in ps_ret))

### 2.3 Cosine vs Euclidean: direction or magnitude?

Two small datasets, one where cosine wins and one where Euclidean wins: short text documents as word-count vectors, and physical sensor readings as coordinates.

**d)** (i) Below are four short documents as word-count vectors over a 6-word vocabulary, two per topic, where one document of each topic is much longer than the other but uses the same words in the same proportions. Before running, predict whether document `A` (short, cooking)'s nearest neighbour will share its topic under Euclidean distance, and under cosine distance. Then run the cell and check. (ii) Below that, three sensor readings (deviations from a baseline, in two channels of the same units) are given: `Q`, `P1` (large deviations, but in the same proportion/direction as `Q`), and `P2` (small deviations, a different direction, but numerically close to `Q`). Before running, predict which of `P1`/`P2` is `Q`'s nearest neighbour under Euclidean, and under cosine. Then run and check. In 1-2 sentences each, explain why the two metrics disagree in each case.

In [ ]:
vocab6 = ["pasta", "sauce", "garlic", "goal", "score", "team"]
docs = {
    "A cooking short": [2, 1, 1, 0, 0, 0],
    "B cooking long":  [10, 5, 5, 0, 0, 0],
    "C sports short":  [0, 0, 0, 2, 1, 1],
    "D sports long":   [0, 0, 0, 10, 5, 5],
}
doc_names, X_doc = list(docs), np.array(list(docs.values()), dtype=float)
D_doc_euc = pairwise_distances(X_doc)
D_doc_cos = 1 - cosine_similarity(X_doc)
i_A = doc_names.index("A cooking short")
print(pd.DataFrame(D_doc_euc, index=doc_names, columns=doc_names).round(2), "\n")
print(pd.DataFrame(D_doc_cos, index=doc_names, columns=doc_names).round(2), "\n")
for name, D in [("Euclidean", D_doc_euc), ("cosine", D_doc_cos)]:
    nn = ...   # skip A itself (index 0 after sorting)
    print(f"{name:9s} nearest neighbour of A: {nn}")

In [ ]:
sensors = {"Q": [1.0, 1.0], "P1": [10.0, 10.0], "P2": [1.5, 0.8]}
sensor_names, X_sensor = list(sensors), np.array(list(sensors.values()))
D_sen_euc = pairwise_distances(X_sensor)
D_sen_cos = 1 - cosine_similarity(X_sensor)
i_Q = sensor_names.index("Q")
print(pd.DataFrame(D_sen_euc, index=sensor_names, columns=sensor_names).round(2), "\n")
print(pd.DataFrame(D_sen_cos, index=sensor_names, columns=sensor_names).round(3))
for name, D in [("Euclidean", D_sen_euc), ("cosine", D_sen_cos)]:
    print(f"{name:9s} nearest neighbour of Q: {sensor_names[np.argsort(D[i_Q])[1]]}")

*Your answer here.*

## 3. K-means and its relatives

K-means looks for $k$ centers minimizing the cost $J=\sum_{j=1}^{k}\sum_{x\in C_j}\lVert x-\mu_j\rVert_2^2$, and Lloyd's algorithm finds a *local* minimum of it. This section reviews why Lloyd's converges but can get stuck, how the initialization helps, how to choose $k$, when the cost function itself is the wrong tool, and a relative, k-medians.

> **Before this section:** complete Worksheet 4 (implementing Lloyd's algorithm).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score, adjusted_rand_score, pairwise_distances
from scipy.spatial.distance import cdist


def kmeans_cost(X, labels):
    """k-means cost J of a labelling, using each cluster's mean as its center (sklearn calls it inertia_)."""
    labels = np.asarray(labels)
    return sum(((X[labels == c] - X[labels == c].mean(axis=0)) ** 2).sum() for c in np.unique(labels))


def show(ax, X, labels, title="", centers=None, s=10):
    """Scatter plot coloured by cluster label; centers as black X's."""
    ax.scatter(X[:, 0], X[:, 1], c=np.asarray(labels) % 10, cmap="tab10", vmin=0, vmax=9, s=s)
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], c="k", marker="X", s=110, edgecolor="w")
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])

### 3.1 Lloyd's algorithm: convergence and local optima

The convergence argument for Lloyd's rests on two facts: the **assignment step** never increases $J$ (each point moves to the center that is best for it), and the **update step** never increases $J$ (for a fixed cluster, the mean minimizes $\sum_{x\in C}\lVert x-\mu\rVert^2$). Since there are finitely many partitions, the algorithm must stop — but not necessarily at the cheapest one.

**a)** Lloyd's always stops, but *where* depends on the start. Below, `KMeans(init="random", n_init=1)` (one Lloyd's run from 6 random data points) is fitted with 200 different seeds on the same data: 6 equal blobs, two isolated and two *close pairs*. What fraction of the runs reaches the best cost? Compare the best- and worst-cost clusterings shown below: what is different about them, and in one sentence, why can Lloyd's end up stuck in the worse one?

In [ ]:
X_lo, _ = datasets.make_blobs(n_samples=[60] * 6, centers=[(0, 0), (3, 0), (10, 10), (13, 10), (0, 10), (10, 0)],
                              cluster_std=0.6, random_state=1)
fits = [KMeans(n_clusters=6, init="random", n_init=1, random_state=s).fit(X_lo) for s in range(200)]
costs = np.array([f.inertia_ for f in fits])
best = costs.min()
print(f"best cost {best:.0f}; runs within 1% of it: {np.mean(costs <= 1.01 * best):.0%}; "
      f"median {np.median(costs) / best:.1f}x best, worst {costs.max() / best:.1f}x best; "
      f"distinct final costs: {len(np.unique(np.round(costs)))}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].hist(costs, bins=np.geomspace(best * 0.95, costs.max() * 1.05, 35), edgecolor="w")
axes[0].set_xscale("log"); axes[0].set_xticks([300, 1000, 3000]); axes[0].set_xticklabels(["300", "1000", "3000"])
axes[0].set_xlabel("final cost (log scale)"); axes[0].set_ylabel("# runs")
axes[0].set_title("200 single runs, random init", fontsize=9)
for ax, (name, i) in zip(axes[1:], [("best", costs.argmin()), ("worst", costs.argmax())]):
    show(ax, X_lo, fits[i].labels_, f"{name} run: cost {costs[i]:.0f}", centers=fits[i].cluster_centers_)
plt.tight_layout(); plt.show()

*Your answer here.*

### 3.2 Initialization: random and k-means++

Both initializations pick $k$ data points, but very differently. With $D(x)$ the distance from $x$ to its closest already-chosen center: **random** picks $k$ points uniformly, ignoring $D$ entirely; **k-means++** picks the first center uniformly at random and then samples each next center with probability proportional to $D(x)^2$ (this is scikit-learn's default, `init="k-means++"`).

**b)** Three datasets: **(A)** 8 tight blobs on a grid plus 3 isolated far-away outliers, $k=8$; **(B)** 9 equal, well-separated blobs, $k=9$; **(C)** 8 well-separated blobs with sizes from 400 down to 15 points, $k=8$. For each dataset and initialization, the cell runs Lloyd's (`KMeans(init=C0, n_init=1)`) from 100 initializations and reports the **success rate**: the fraction of runs within 1% of the best cost found by either method. (i) Before running, predict whether k-means++ beats random on each dataset. Look especially at the **min cost / best** column: on which dataset would you expect random's *best* run out of 100 to still fall well short of the true best, unlike on the other two? (ii) Explain the mechanism from the min/max columns: what quantity does uniform random sampling of *points* favor, what quantity does k-means++'s $D(x)^2$ weighting favor instead, and why does that difference show up so much more sharply on that one dataset?

In [ ]:
grid_A = [(12 * c, 12 * r) for c in range(4) for r in range(2)]
Xb, yb = datasets.make_blobs(n_samples=[60] * 8, centers=grid_A, cluster_std=1.0, random_state=0)
X_A = np.vstack([Xb, [[18.0, 32.0], [56.0, -10.0], [-16.0, 6.0]]])           # 3 outliers
X_B, _ = datasets.make_blobs(n_samples=[60] * 9, centers=[(10 * c, 10 * r) for c in range(3) for r in range(3)],
                             cluster_std=1.0, random_state=0)
X_C, _ = datasets.make_blobs(n_samples=[400, 15, 200, 30, 100, 15, 50, 30],
                             centers=[(8 * c, 8 * r) for c in range(4) for r in range(2)], cluster_std=1.0, random_state=0)
DATA = {"A: outliers": (X_A, 8), "B: clean blobs": (X_B, 9), "C: unequal sizes": (X_C, 8)}
INITS = {"random": "random", "k-means++": "k-means++"}

runs, rows = {}, []
for dname, (X, k) in DATA.items():
    for m, spec in INITS.items():
        runs[dname, m] = [KMeans(n_clusters=k, init=spec, n_init=1, random_state=s).fit(X) for s in range(100)]
    best = min(f.inertia_ for m in INITS for f in runs[dname, m])
    for m in INITS:
        c = np.array([f.inertia_ for f in runs[dname, m]]) / best
        rows.append({"dataset": dname, "init": m, "success rate": np.mean(c <= 1.01),
                     "min cost / best": c.min(), "max cost / best": c.max()})
table = pd.DataFrame(rows).set_index(["dataset", "init"])
print(table.round(2).to_string())

# a typical (median-cost) run of EACH method on each dataset
fig, axes = plt.subplots(2, 3, figsize=(12, 6.4))
for col, dname in zip(range(3), DATA):
    for row, m in enumerate(INITS):
        f = sorted(runs[dname, m], key=lambda f: f.inertia_)[50]
        show(axes[row, col], DATA[dname][0], f.labels_, f"{dname}: median {m} run", centers=f.cluster_centers_)
plt.tight_layout(); plt.show()

*Your answer here.*

### 3.3 Choosing $k$: elbow and silhouette

Since the optimal $J$ can only decrease as $k$ grows, "lowest cost" cannot pick $k$ on its own. The **elbow** method looks for the point of diminishing returns purely by eye: plot $J(k)$ against $k$ and look for the bend where the curve stops dropping quickly.

**c)** The dataset below has three far-apart groups, each made of two nearby sub-blobs. Looking only at the *shape* of the curve, where does it bend? Is $k=3$ or $k=6$ the "right" elbow? Give an application in which each would be correct, and say what information from *outside* the data decides.

In [ ]:
groups, sub = np.array([[0, 0], [10, 0], [5, 8.5]]), np.array([[-1.3, 0], [1.3, 0]])
X_h, y_h = datasets.make_blobs(n_samples=600, centers=np.array([g + o for g in groups for o in sub]),
                               cluster_std=0.45, random_state=1)
ks = np.arange(1, 11)
J = np.array([KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_h).inertia_ for k in ks])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
show(axes[0], X_h, y_h, "3 groups x 2 sub-blobs (true sub-blob labels)")
axes[1].plot(ks, J, "o-"); axes[1].set_yscale("log"); axes[1].set_xlim(0.5, 10.8)
axes[1].set_xlabel("k"); axes[1].set_ylabel("J(k), log scale"); axes[1].set_title("elbow curve", fontsize=9)
plt.tight_layout(); plt.show()

*Your answer here.*

For each point $i$, the **silhouette** compares $a_i$, the mean distance to the other members of its own cluster, with $b_i$, the mean distance to the *nearest other* cluster:

$$a_i = \frac{1}{|C(i)|-1}\sum_{j \in C(i),\, j \ne i} d(i,j), \qquad b_i = \min_{C \ne C(i)} \frac{1}{|C|}\sum_{j \in C} d(i,j), \qquad s_i = \frac{b_i - a_i}{\max(a_i, b_i)},$$

with $s_i = 0$ for a point that is alone in its cluster. **Vectorization:** with the $n\times n$ distance matrix $D$ and the $n\times K$ one-hot matrix $H$ ($H_{jc}=1$ if point $j$ is in cluster $c$), $(DH)_{ic}$ is the *sum* of distances from $i$ to cluster $c$; divide column $c$ by $|C_c|$ for a mean (by $|C(i)|-1$ for $i$'s own cluster, since $d(i,i)=0$).

**d)** (i) Complete `silhouette_samples_scratch` (no loops over points) and run the check against scikit-learn.

In [ ]:
def silhouette_samples_scratch(D, labels):
    """Silhouette s_i of every point, given a precomputed (n, n) distance matrix D. Singletons get s_i = 0."""
    labels = np.asarray(labels)
    n = len(labels)
    clusters, idx = np.unique(labels, return_inverse=True)                  # idx[i] = column of i's own cluster
    H = (idx[:, None] == np.arange(len(clusters))[None, :]).astype(float)   # one-hot, (n, K)
    sizes = H.sum(axis=0)                                                    # |C_c|
    sums = D @ H                                                             # sums[i, c] = sum of d(i, j) over j in C_c
    own_size = sizes[idx]
    # YOUR CODE HERE
    ...
    return s

In [ ]:
# Check your work against sklearn (including a cluster with a single point)
X5, _ = datasets.make_blobs(n_samples=400, centers=5, cluster_std=0.9, random_state=7)
lab5 = KMeans(n_clusters=5, n_init=10, random_state=0).fit(X5).labels_
D5 = pairwise_distances(X5)
assert np.allclose(silhouette_samples_scratch(D5, lab5), silhouette_samples(X5, lab5))
lab_single = lab5.copy(); lab_single[0] = 99
assert np.allclose(silhouette_samples_scratch(D5, lab_single), silhouette_samples(X5, lab_single))
print(f"silhouette matches sklearn; mean silhouette on 5 blobs, k=5: {silhouette_samples_scratch(D5, lab5).mean():.3f}")

(ii) On two moons and two concentric circles, the next cell scores two labellings with the mean silhouette: the true labels and k-means' answer for $k=2$; it also prints the k-means silhouette for $k=2,\dots,8$. Before running, guess which labelling scores higher. Then: the silhouette prefers a labelling that is clearly "wrong". Wrong according to what? Argue that the silhouette is doing exactly its job, and describe what it rewards.

In [ ]:
X_moons, y_moons = datasets.make_moons(n_samples=400, noise=0.06, random_state=0)
X_circ, y_circ = datasets.make_circles(n_samples=400, factor=0.4, noise=0.05, random_state=0)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for j, (name, X, y) in enumerate([("moons", X_moons, y_moons), ("circles", X_circ, y_circ)]):
    D = pairwise_distances(X)
    lab = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X).labels_
    s_true, s_km = silhouette_samples_scratch(D, y).mean(), silhouette_samples_scratch(D, lab).mean()
    sweep = [silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=0).fit(X).labels_) for k in range(2, 9)]
    print(f"{name:8s} silhouette: truth {s_true:.2f} | k-means k=2 {s_km:.2f} | k-means k=2..8 {np.round(sweep, 2)}")
    show(axes[2 * j], X, y, f"{name}: truth, silhouette {s_true:.2f}")
    show(axes[2 * j + 1], X, lab, f"{name}: k-means k=2, silhouette {s_km:.2f}")
plt.tight_layout(); plt.show()

*Your answer here.*

### 3.4 Failure modes: the optimizer or the objective?

When k-means disagrees with the true labels, there are two possible reasons. **Optimizer failure:** the truth has *lower* cost, but Lloyd's got stuck in a local optimum as in a); more restarts help. **Objective failure:** the truth has *higher* cost than k-means' answer, so even a perfect optimizer would return the "wrong" clustering. The diagnostic is to compare $J(\text{truth})$ with $J(\text{k-means})$. We also report the ARI (adjusted Rand index: 1 = matches the true labels perfectly, about 0 = no better than random).

**e)** Four datasets: three blobs **stretched** by the same linear map into long parallel ellipses; **unequal spread** (a diffuse cluster with $\sigma=2.5$ next to a tight one with $\sigma=0.4$); **unequal sizes** (600 points next to two clusters of 25); and the concentric **circles**. Before running, predict for each dataset whether $J(\text{truth})$ will be larger or smaller than $J(\text{k-means})$ (with the correct $k$ and 10 restarts). Then explain, through the cost function, why k-means' answer wins in each case, and whether `n_init=1000` or an exact solver would recover the truth.

In [ ]:
X_s0, y_str = datasets.make_blobs(n_samples=450, centers=3, cluster_std=1.0, random_state=170)
X_str = X_s0 @ np.array([[0.6, -0.6], [-0.4, 0.8]])
rng = np.random.default_rng(1)
X_var = np.vstack([rng.normal([0, 0], 2.5, (300, 2)), rng.normal([6.5, 0], 0.4, (100, 2))])
y_var = np.repeat([0, 1], [300, 100])
X_size = np.vstack([rng.normal([0, 0], 1.5, (600, 2)), rng.normal([5, 3], 0.5, (25, 2)), rng.normal([5, -3], 0.5, (25, 2))])
y_size = np.repeat([0, 1, 2], [600, 25, 25])
cases = {"stretched": (X_str, y_str), "unequal spread": (X_var, y_var),
         "unequal sizes": (X_size, y_size), "circles": (X_circ, y_circ)}

rows = {}
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for j, (name, (X, y)) in enumerate(cases.items()):
    lab = KMeans(n_clusters=len(np.unique(y)), n_init=10, random_state=0).fit(X).labels_
    J_true = ...
    J_km = ...
    rows[name] = {"ARI(k-means)": adjusted_rand_score(y, lab), "J(truth)": J_true, "J(k-means)": J_km,
                  "J(truth) / J(k-means)": J_true / J_km}
    show(axes[0, j], X, y, f"{name}: truth, J={J_true:.0f}")
    show(axes[1, j], X, lab, f"k-means: J={J_km:.0f}, ARI={rows[name]['ARI(k-means)']:.2f}")
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).T.round(2).to_string())

*Your answer here.*

### 3.5 A relative: k-medians

**k-medians** minimizes $J_1=\sum_{j}\sum_{x\in C_j}\lVert x-m_j\rVert_1$ with the same alternation: assign each point to the center with the smallest $L_1$ distance, then move each center to the **coordinate-wise median** of its cluster (the median minimizes a sum of absolute deviations, just as the mean minimizes a sum of squared deviations).

**f)** Complete `kmedians`. The next cell adds 15 far outliers at distance $d$ to the right of one of three blobs and runs k-means and k-medians from the *same* initial centers, for $d=0,8,16,30$. **Centroid drift** is, over the three true centers, the largest distance to the nearest found center. Before running, predict how the drift of each method behaves as $d$ grows, and explain the result afterwards using the per-$d$ pictures.

In [ ]:
def kmedians(X, init_centers, max_iter=100):
    """k-medians from init_centers (k, d): L1 assignment, coordinate-wise median update.
    Stops when the labels no longer change; an empty cluster keeps its old center. Returns (centers, labels)."""
    C = np.array(init_centers, dtype=float)
    labels = None
    for _ in range(max_iter):
        # YOUR CODE HERE
        ...
    return C, labels

In [ ]:
true_centers = np.array([[0, 0], [5, 0], [2.5, 4.5]])
X_blob, y_blob = datasets.make_blobs(n_samples=300, centers=true_centers, cluster_std=0.8, random_state=0)
init3 = X_blob[np.random.default_rng(0).choice(300, 3, replace=False)]
noise15 = np.random.default_rng(1).normal(0, 1, (15, 2))

# check your work
C, lab = kmedians(X_blob, init3)
assert np.array_equal(lab, cdist(X_blob, C, "cityblock").argmin(axis=1)), "labels must be the nearest-L1 center"
assert all(np.allclose(C[j], np.median(X_blob[lab == j], axis=0)) for j in range(3)), "centers must be medians"


def drift(found):
    """Largest distance from a true center to the nearest found center."""
    return cdist(true_centers, found).min(axis=1).max()


def run_both(d):
    Xo = np.vstack([X_blob, true_centers[1] + [d, 0] + noise15])
    km = KMeans(n_clusters=3, init=init3, n_init=1).fit(Xo)
    return Xo, (km.cluster_centers_, km.labels_), kmedians(Xo, init3)


ds = [0, 8, 16, 30]
print("d        :", ds)
print("k-means  :", [round(drift(run_both(d)[1][0]), 2) for d in ds])
print("k-medians:", [round(drift(run_both(d)[2][0]), 2) for d in ds])

fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for col, d in enumerate(ds):
    Xo, (C_km, lab_km), (C_md, lab_md) = run_both(d)
    show(axes[0, col], Xo, lab_km, f"d={d} k-means: drift {drift(C_km):.2f}", centers=C_km)
    show(axes[1, col], Xo, lab_md, f"d={d} k-medians: drift {drift(C_md):.2f}", centers=C_md)
axes[0, 0].set_ylabel("k-means"); axes[1, 0].set_ylabel("k-medians")
plt.tight_layout(); plt.show()

*Your answer here.*

**g)** The center choice changes which "true" clustering k-means and k-medians recover. Below are four small datasets: two where the mean is genuinely the better center (Gaussian, roughly spherical clusters, no outliers), and two where the coordinate-wise median is better (a minority of far outliers, or a skewed/heavy-tailed spread that drags the mean off the bulk of the cluster). For each, before running, predict which method will match the true labels better; then verify with the ARI, and briefly say why.

In [ ]:
def fit_both(X, k, seed):
    """Fit k-means and k-medians from the SAME random init and return (labels_km, labels_md)."""
    init = X[np.random.default_rng(seed).choice(len(X), k, replace=False)]
    km = KMeans(n_clusters=k, init=init, n_init=1).fit(X)
    _, lab_md = kmedians(X, init)
    return km.labels_, lab_md


# (1) small, close, spherical Gaussian blobs, no outliers: the mean is a more precise center than the median
X1, y1 = datasets.make_blobs(n_samples=[15] * 4, centers=[(0, 0), (2.6, 0), (0, 2.6), (2.6, 2.6)],
                             cluster_std=0.9, random_state=7)
# (2) spherical Gaussian blobs, more of them and even smaller, same mechanism
X2, y2 = datasets.make_blobs(n_samples=[12] * 5, centers=[(6 * np.cos(t), 6 * np.sin(t)) for t in np.linspace(0, 2 * np.pi, 5, endpoint=False)],
                             cluster_std=1.1, random_state=0)
# (3) two blobs, each with a handful of far outliers dragging the mean toward the other cluster
Xb3, y3b = datasets.make_blobs(n_samples=[90, 90], centers=[(0, 0), (4, 0)], cluster_std=0.7, random_state=5)
rng3 = np.random.default_rng(5)
out3 = np.vstack([rng3.normal([0, 8], 0.6, (10, 2)), rng3.normal([4, -8], 0.6, (10, 2))])
X3, y3 = np.vstack([Xb3, out3]), np.concatenate([y3b, [0] * 10 + [1] * 10])
# (4) two blobs with a one-sided heavy (skewed) tail instead of point outliers
Xb4, y4b = datasets.make_blobs(n_samples=[90, 90], centers=[(0, 0), (4, 0)], cluster_std=0.6, random_state=6)
rng4 = np.random.default_rng(6)
tail4 = np.vstack([rng4.exponential(5, 20)[:, None] * [0, 1] + [0, 1.5], rng4.exponential(5, 20)[:, None] * [0, -1] + [4, -1.5]])
X4, y4 = np.vstack([Xb4, tail4]), np.concatenate([y4b, [0] * 20 + [1] * 20])
cases2 = {"(1) close spherical": (X1, y1, 4, 7), "(2) tiny spherical": (X2, y2, 5, 7),
          "(3) point outliers": (X3, y3, 2, 0), "(4) skewed tail": (X4, y4, 2, 0)}

rows2 = {}
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for j, (name, (X, y, k, seed)) in enumerate(cases2.items()):
    lab_km, lab_md = fit_both(X, k, seed)
    rows2[name] = {"ARI(k-means)": adjusted_rand_score(y, lab_km), "ARI(k-medians)": adjusted_rand_score(y, lab_md)}
    show(axes[0, j], X, lab_km, f"{name}\nk-means ARI={rows2[name]['ARI(k-means)']:.2f}")
    show(axes[1, j], X, lab_md, f"k-medians ARI={rows2[name]['ARI(k-medians)']:.2f}")
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows2).T.round(2).to_string())

*Your answer here.*

## 4. Hierarchical clustering

Agglomerative clustering starts with every point as its own cluster and repeatedly merges the two closest clusters; the merge history (a dendrogram) can be cut at any level to give a flat clustering. The result depends on three choices: the cluster distance $D$ (the linkage), the point distance $d$, and where you cut.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import adjusted_rand_score

### 4.1 Cluster distances $D(C_1, C_2)$

The algorithm is always "merge the two closest clusters"; only $D$ changes. For clusters $A$, $B$ with centroids $\mu_A$, $\mu_B$:

- **single:** $D(A,B) = \min_{a\in A,\, b\in B} d(a,b)$
- **complete:** $D(A,B) = \max_{a\in A,\, b\in B} d(a,b)$
- **average:** $D(A,B) = \dfrac{1}{\lvert A\rvert\,\lvert B\rvert}\displaystyle\sum_{a\in A}\sum_{b\in B} d(a,b)$
- **centroid:** $D(A,B) = \lVert \mu_A - \mu_B \rVert_2$
- **Ward:** $D(A,B) = \mathrm{SSE}(A\cup B) - \mathrm{SSE}(A) - \mathrm{SSE}(B)$, where $\mathrm{SSE}(C) = \sum_{x\in C}\lVert x-\mu_C\rVert_2^2$

**a)** Fill in the five cluster-distance functions. Each receives two lists of point indices `A`, `B`, the $n \times n$ point-distance matrix `Dm` (so $d$ is pluggable) and the data `X` (needed only by centroid and Ward, which are defined through means).

In [ ]:
def single_link(A, B, Dm, X):
    """Minimum point distance between a point of A and a point of B."""
    # YOUR CODE HERE
    ...

def complete_link(A, B, Dm, X):
    """Maximum point distance between a point of A and a point of B."""
    # YOUR CODE HERE
    ...

def average_link(A, B, Dm, X):
    """Average of all |A|*|B| point distances between A and B."""
    # YOUR CODE HERE
    ...

def centroid_link(A, B, Dm, X):
    """Euclidean distance between the centroids of A and B (ignores Dm)."""
    # YOUR CODE HERE
    ...

def sse(P):
    """Sum of squared Euclidean distances of the rows of P to their mean."""
    # YOUR CODE HERE
    ...

def ward_link(A, B, Dm, X):
    """Increase in SSE caused by merging A and B (ignores Dm)."""
    # YOUR CODE HERE
    ...

CLUSTER_DIST = {"single": single_link, "complete": complete_link, "average": average_link,
                "centroid": centroid_link, "ward": ward_link}

**Check your work.** The provided loop below is the whole agglomerative algorithm (naive, but correct): it plugs in your $D$ and returns the $n-1$ merge heights, which must match scipy's `linkage`. (scipy reports Ward heights as $\sqrt{2\,\Delta\mathrm{SSE}}$; we compare sorted heights because ties may be broken in a different order.)

In [ ]:
def agglomerative_heights(X, D):
    """Provided: merge the two closest clusters until one is left; return the merge heights."""
    Dm = squareform(pdist(X))
    clusters = [[i] for i in range(len(X))]
    heights = []
    while len(clusters) > 1:
        h, i, j = min((D(clusters[i], clusters[j], Dm, X), i, j)
                      for i, j in itertools.combinations(range(len(clusters)), 2))
        heights.append(h)
        clusters[i] = clusters[i] + clusters[j]
        del clusters[j]
    return np.array(heights)

rng = np.random.default_rng(1)
Xc = np.r_[rng.normal(0, 1, (15, 2)), rng.normal([5, 2], 0.7, (12, 2))]
for name, D in CLUSTER_DIST.items():
    h = agglomerative_heights(Xc, D)
    h = np.sqrt(2 * h) if name == "ward" else h
    assert np.allclose(np.sort(h), np.sort(linkage(Xc, method=name)[:, 2])), name
    print(f"{name:9s} merge heights match scipy")

**b)** Take $A=(0,0)$, $B=(1,1)$, $C=(3,0)$, $D=(0,-2)$ with Euclidean $d$. Before running the next cell, predict for each of the five linkages whether $C$ or $D$ is closer to the cluster $\{A,B\}$. For two of the linkages the answer is a tie — can you spot them without computing?

*Your answer here.*

In [ ]:
P4 = np.array([[0, 0], [1, 1], [3, 0], [0, -2]], dtype=float)   # A, B, C, D
D4 = squareform(pdist(P4))
print(pd.DataFrame({name: {"C to {A,B}": f([0, 1], [2], D4, P4), "D to {A,B}": f([0, 1], [3], D4, P4)}
                    for name, f in CLUSTER_DIST.items()}).T.round(3))

### 4.2 The data decides: which linkage?

Six datasets designed to bring out each linkage's behaviour. We cut each tree into the true number of clusters $k$ with `fcluster(Z, k, criterion="maxclust")` and score it with the adjusted Rand index (ARI: 1 = matches the true labels perfectly, about 0 = no better than random).

| # | dataset | $k$ |
|---|---|---|
| 1 | two long parallel strips | 2 |
| 2 | two concentric circles | 2 |
| 3 | two blobs joined by a thin bridge of points | 2 |
| 4 | a large diffuse disk next to a small tight blob | 2 |
| 5 | three globular blobs with a few outliers | 3 |
| 6 | one big spread-out cluster plus two small tight ones | 3 |

In [ ]:
def disk(rng, n, center, radius):
    th = rng.uniform(0, 2 * np.pi, n); r = radius * np.sqrt(rng.uniform(0, 1, n))
    return np.c_[center[0] + r * np.cos(th), center[1] + r * np.sin(th)]

def make_strips(rng):
    x = rng.uniform(0, 10, 200); y = np.r_[np.zeros(100), np.full(100, 1.5)] + rng.normal(0, 0.12, 200)
    return np.c_[x, y], np.repeat([0, 1], 100)

def make_circles_(rng):
    return datasets.make_circles(n_samples=200, factor=0.45, noise=0.04, random_state=1)

def make_bridge(rng):
    a = rng.normal([0, 0], 0.6, (100, 2)); b = rng.normal([6, 0], 0.6, (100, 2))
    t = np.linspace(1.2, 4.8, 16); br = np.c_[t, rng.normal(0, 0.05, 16)]
    return np.r_[a, b, br], np.r_[np.zeros(100), np.ones(100), t > 3].astype(int)   # bridge: nearer blob

def make_big_small(rng):
    return np.r_[disk(rng, 250, (0, 0), 2.5), rng.normal([4.2, 0], 0.25, (30, 2))], np.repeat([0, 1], [250, 30])

def make_outliers(rng):
    C = np.array([[0, 0], [5, 0], [2.5, 4.3]])
    X, y = datasets.make_blobs(n_samples=240, centers=C, cluster_std=0.7, random_state=3)
    ci = np.array([0, 1, 2, 0])                                   # 4 outliers, each pushed out from a blob
    phi = np.arctan2(*(C[ci] - C.mean(0)).T[::-1]) + rng.uniform(-0.5, 0.5, 4)
    return np.r_[X, C[ci] + 4.0 * np.c_[np.cos(phi), np.sin(phi)]], np.r_[y, ci]

def make_unequal(rng):
    a = rng.normal([0, 0], 1.5, (300, 2)); b = rng.normal([5.5, 2.5], 0.4, (20, 2)); c = rng.normal([5.5, -2.5], 0.4, (20, 2))
    return np.r_[a, b, c], np.repeat([0, 1, 2], [300, 20, 20])

SCENARIOS = {"1 strips": (make_strips, 0, 2), "2 circles": (make_circles_, 0, 2),
             "3 bridge": (make_bridge, 0, 2), "4 big diffuse + small tight": (make_big_small, 1, 2),
             "5 blobs + outliers": (make_outliers, 0, 3), "6 unequal sizes": (make_unequal, 0, 3)}
DATA = {name: (*f(np.random.default_rng(seed)), k) for name, (f, seed, k) in SCENARIOS.items()}

**c)** For each of the six datasets, predict which of single / complete / average / centroid / Ward recovers the ground truth and which fail. Where you predict a failure, say *how* it fails (which clusters get merged or split). Then run the cell and check your predictions.

*Your answer here.*

In [ ]:
METHODS = ["single", "complete", "average", "centroid", "ward"]
results = {}
fig, axes = plt.subplots(len(DATA), 1 + len(METHODS), figsize=(14, 13))
for r, (name, (X, y, k)) in enumerate(DATA.items()):
    axes[r, 0].scatter(X[:, 0], X[:, 1], c=y, cmap="tab10", s=5, vmin=0, vmax=9)
    axes[r, 0].set_ylabel(name, fontsize=9); axes[r, 0].set_title("ground truth", fontsize=9)
    for c, method in enumerate(METHODS, start=1):
        Z = linkage(X, method=method)
        labels = fcluster(Z, k, criterion="maxclust")
        results[(name, method)] = adjusted_rand_score(y, labels)
        axes[r, c].scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=5, vmin=1, vmax=10)
        axes[r, c].set_title(f"{method}: ARI={results[(name, method)]:.2f}", fontsize=9)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout(); plt.show()
print(pd.Series(results).unstack()[METHODS].loc[list(DATA)].round(2))

**d)** Explain two of the results mechanistically, using the definitions of $D$: (i) why single link merges the two blobs of the **bridge** dataset, while complete and average link are not fooled; (ii) why complete link cuts the **big diffuse** disk in half even though there is a visible gap between the disk and the small blob.

*Your answer here.*

### 4.4 Cutting the dendrogram

The data below has structure at two scales: two far-apart groups, each made of two nearby blobs (4 blobs in total). A common heuristic cuts the tree where the jump between consecutive merge heights is largest.

**e)** Complete the blank that turns the biggest gap into a suggested $k$. The heuristic suggests $k=2$, yet the data was generated from 4 blobs. Which answer is right, and what does the second-largest gap tell you?

In [ ]:
C_nest = np.array([[0, 0], [4, 0], [0, 12], [4, 12]])
X_nest, y_nest = datasets.make_blobs(n_samples=200, centers=C_nest, cluster_std=0.5, random_state=0)
Z_nest = linkage(X_nest, method="average")
h, n = Z_nest[:, 2], len(X_nest)
t_big = n - 11 + np.argmax(np.diff(h[-10:]))          # merge index just BELOW the biggest jump (last 10 merges)
k_suggested = ...
cut = (h[t_big] + h[t_big + 1]) / 2

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(np.arange(n - 1), h, ".-", ms=3); axes[0].axhline(cut, color="r", ls="--", lw=1)
axes[0].set_xlabel("merge step $t$"); axes[0].set_ylabel("merge height")
axes[0].set_title(f"average link: biggest gap suggests k = {k_suggested}", fontsize=10)
dendrogram(Z_nest, ax=axes[1], no_labels=True, color_threshold=cut)
axes[1].axhline(cut, color="r", ls="--", lw=1); axes[1].axhline(np.mean(h[-4:-2]), color="gray", ls=":", lw=1)
axes[1].set_title("red = cut at biggest gap, gray = cut giving 4", fontsize=10)
plt.tight_layout(); plt.show()
print("last 5 merge heights:", h[-5:].round(2))
for k in [2, 4]:
    print(f"k={k}: ARI vs the 4 generating blobs = {adjusted_rand_score(y_nest, fcluster(Z_nest, k, criterion='maxclust')):.2f}")

*Your answer here.*

### 4.5 Cost: why hierarchical clustering doesn't scale like k-means

**f)** (i) The naive algorithm stores the $n\times n$ distance matrix and, at each of its $n-1$ merges, scans the cluster-distance matrix for the smallest entry. What are its memory and time costs in $O(\cdot)$? (ii) Complete the blank to compute the memory (float64, 8 bytes per number) of the $\binom{n}{2}$ distinct distances, and read off the answer for $n=10^5$ and $n=10^6$. What does this mean in practice, compared with k-means? The timings of scipy's (much smarter, compiled) `linkage` are shown for reference.

In [ ]:
for n in [10**3, 10**4, 10**5, 10**6]:
    gb = ...
    print(f"n = {n:>9,}: distance matrix needs {gb:10.3f} GB")

for n in [1000, 2000, 4000]:
    Xt = np.random.default_rng(0).normal(size=(n, 2))
    t0 = time.perf_counter(); linkage(Xt, method="average")
    print(f"scipy linkage, n = {n}: {time.perf_counter() - t0:.3f} s")

*Your answer here.*

## 5. Density-based clustering: DBSCAN

DBSCAN asks one local question about every point — "is the region around me crowded?" — and lets clusters be whatever connected crowded regions exist, of any shape, with everything else declared **noise**. This section uses scikit-learn's `DBSCAN` to pin down its conventions, tune $\varepsilon$ and $\text{minPts}$, and map out where it beats k-means and where it loses.

> **Before this section:** complete Worksheet 6 (implementing DBSCAN).

In [ ]:
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score, pairwise_distances


def point_types(model):
    """'core' / 'border' / 'noise' for every point, read off a fitted sklearn DBSCAN."""
    kind = np.where(model.labels_ == -1, "noise", "border").astype(object)
    kind[model.core_sample_indices_] = "core"
    return kind


def plot_labels(ax, X, labels, title="", kind=None, s=14):
    """Clusters in tab10, noise as gray x's. If kind is given, border points are drawn as hollow triangles."""
    labels = np.asarray(labels)
    noise = labels == -1
    ax.scatter(X[noise, 0], X[noise, 1], c="lightgray", marker="x", s=s)
    full = ~noise if kind is None else (kind == "core")
    ax.scatter(X[full, 0], X[full, 1], c=labels[full] % 10, cmap="tab10", vmin=0, vmax=9, s=s)
    if kind is not None:
        b = kind == "border"
        ax.scatter(X[b, 0], X[b, 1], facecolors="none", marker="^", s=2.5 * s,
                   edgecolors=plt.cm.tab10((labels[b] % 10) / 9.999))
    ax.set_title(title, fontsize=8)
    ax.set_aspect("equal", adjustable="datalim"); ax.set_xticks([]); ax.set_yticks([])


def n_clusters(labels):
    return len(set(labels)) - (1 if -1 in labels else 0)


def ari(y_true, y_pred):
    """Adjusted Rand index, with every noise point (-1) treated as its own singleton cluster."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    for a in (y_true, y_pred):
        m = a.max() + 1
        a[a == -1] = np.arange(m, m + (a == -1).sum())
    return adjusted_rand_score(y_true, y_pred)

### 5.1 Core, border and noise in scikit-learn

With $N_\varepsilon(x) = \{\, y : d(x,y) \le \varepsilon \,\}$: $x$ is **core** if $\lvert N_\varepsilon(x)\rvert \ge \text{minPts}$, **border** if it is not core but lies in the $\varepsilon$-neighborhood of a core point, and **noise** otherwise. Since $d(x,x)=0$, this $N_\varepsilon(x)$ contains $x$ itself — and that is sklearn's convention: `min_samples` counts the point itself.

**a)** A classmate's Worksheet 6 implementation calls $x$ core if there are at least $\text{minPts}$ *other* points within $\varepsilon$. Which `min_samples` makes sklearn reproduce their core points for $\text{minPts}=5$? Fill it in, then use the figure to check one point of each kind against the definitions (count the points inside each dashed $\varepsilon$-disk).

In [ ]:
rng = np.random.default_rng(1)
X_m, _ = datasets.make_moons(200, noise=0.08, random_state=1)
X_demo = np.vstack([X_m, np.column_stack([rng.uniform(-1.5, 2.5, 16), rng.uniform(-1.0, 1.5, 16)])])


def classmate_is_core(X, eps, min_pts):
    """The classmate's rule: core iff at least min_pts OTHER points lie within eps."""
    n_within = (pairwise_distances(X) <= eps).sum(axis=1)      # includes the point itself
    return n_within - 1 >= min_pts


eps_demo, mp_classmate = 0.2, 5
model = DBSCAN(eps=eps_demo, min_samples=...).fit(X_demo)
kind = point_types(model)
print("same core points as the classmate:", np.array_equal(kind == "core", classmate_is_core(X_demo, eps_demo, mp_classmate)))
print({k: int((kind == k).sum()) for k in ["core", "border", "noise"]})

counts = (pairwise_distances(X_demo) <= eps_demo).sum(axis=1)
picks = {"core": np.flatnonzero(kind == "core")[np.argmin(counts[kind == "core"])],   # least crowded core point
         "border": np.flatnonzero(kind == "border")[0],
         "noise": np.flatnonzero(kind == "noise")[np.argmax(counts[kind == "noise"])]}  # most crowded noise point
fig, ax = plt.subplots(figsize=(7, 4))
plot_labels(ax, X_demo, model.labels_, kind=kind, s=14,
            title=f"eps={eps_demo}, min_samples={model.min_samples}: filled = core, triangle = border, x = noise")
for name, i in picks.items():
    ax.add_patch(plt.Circle(X_demo[i], eps_demo, fill=False, ls="--", color="k"))
    ax.annotate(f"{name}\n|N|={counts[i]}", X_demo[i], xytext=(8, 8), textcoords="offset points", fontsize=8)
plt.show()

*Your answer here.*

### 5.2 Choosing $\varepsilon$ and $\text{minPts}$

The test dataset: two moons, a small tight blob and 50 uniform noise points (true label $-1$). ARI: 1 = matches the true labels perfectly, about 0 = no better than random (our `ari` helper counts each noise point as its own cluster, so calling a real cluster "noise" is penalized).

**b)** Before running, predict which corner of the grid ($\varepsilon$ down the rows, $\text{minPts}$ across) is almost all noise and which is one giant cluster. After running, use the curves to name the four regimes as $\varepsilon$ grows for $\text{minPts}=6$ (with rough $\varepsilon$ ranges), and explain why the number of clusters first *rises* and then falls.

In [ ]:
rng = np.random.default_rng(6)
X_mo, y_mo = datasets.make_moons(400, noise=0.06, random_state=6)
X_sw = np.vstack([X_mo, rng.normal([2.6, 1.0], 0.15, (80, 2)),
                  np.column_stack([rng.uniform(-1.5, 3.3, 50), rng.uniform(-1.0, 1.6, 50)])])
y_sw = np.r_[y_mo, np.full(80, 2), np.full(50, -1)]

eps_grid, mp_grid = [0.04, 0.1, 0.2, 0.5], [3, 6, 12, 30]
fig, axes = plt.subplots(4, 4, figsize=(10, 7.2))
for r, eps in enumerate(eps_grid):
    for c, mp in enumerate(mp_grid):
        lab = ...
        plot_labels(axes[r, c], X_sw, lab, s=4,
                    title=f"eps={eps}, mp={mp}\nk={n_clusters(lab)}, noise={np.sum(lab == -1)}, ARI={ari(y_sw, lab):.2f}")
fig.tight_layout(); plt.show()

eps_range = np.round(np.arange(0.02, 0.61, 0.01), 2)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for mp in mp_grid:
    labs = [DBSCAN(eps=e, min_samples=mp).fit_predict(X_sw) for e in eps_range]
    axes[0].plot(eps_range, [n_clusters(l) for l in labs], label=f"minPts={mp}")
    axes[1].plot(eps_range, [np.sum(l == -1) for l in labs])
    axes[2].plot(eps_range, [ari(y_sw, l) for l in labs])
axes[0].set_ylabel("# clusters"); axes[0].set_ylim(0, 25); axes[0].axhline(3, color="k", ls=":"); axes[0].legend(fontsize=7)
axes[1].set_ylabel("# noise points"); axes[1].axhline(50, color="k", ls=":")
axes[2].set_ylabel("ARI vs truth")
for ax in axes: ax.set_xlabel(r"$\varepsilon$")
fig.tight_layout(); plt.show()
for e in [0.05, 0.08, 0.15, 0.25, 0.3, 0.35]:
    lab = DBSCAN(eps=e, min_samples=6).fit_predict(X_sw)
    print(f"minPts=6, eps={e}: {n_clusters(lab):2d} clusters, {np.sum(lab == -1):3d} noise, ARI={ari(y_sw, lab):.2f}")

*Your answer here.*

### 5.3 The data decides

For each dataset k-means gets the *true* number of clusters; DBSCAN gets a hand-picked $(\varepsilon, \text{minPts})$ chosen from a $k$-distance plot and a small sweep, as above.

**c)** Four datasets: (1) two moons, (2) two concentric circles, (3) a "smiley" (ring face, two eyes, curved mouth), (4) three Gaussian blobs buried in 300 uniform noise points. Before running, predict which method wins each and what k-means's mistake will look like. After running, explain each result through the two algorithms' *definitions of a cluster*, and say where DBSCAN's remaining disagreement on the noisy blobs comes from.

In [ ]:
def smiley(seed=0):
    rng = np.random.default_rng(seed)
    t = rng.uniform(0, 2 * np.pi, 400)
    face = np.c_[3 * np.cos(t), 3 * np.sin(t)] + rng.normal(0, 0.1, (400, 2))
    eyes = np.vstack([rng.normal([-1, 1], 0.2, (60, 2)), rng.normal([1, 1], 0.2, (60, 2))])
    t = rng.uniform(np.deg2rad(200), np.deg2rad(340), 150)
    mouth = np.c_[1.7 * np.cos(t), 1.7 * np.sin(t)] + rng.normal(0, 0.08, (150, 2))
    return np.vstack([face, eyes, mouth]), np.repeat([0, 1, 2, 3], [400, 60, 60, 150])


def noisy_blobs(seed=2):
    rng = np.random.default_rng(seed)
    Xb, yb = datasets.make_blobs(450, centers=[[-2, -2], [2, -1], [0, 2.5]], cluster_std=0.5, random_state=seed)
    return np.vstack([Xb, rng.uniform(-6, 6, (300, 2))]), np.r_[yb, np.full(300, -1)]


wins = {  # name: (X, y, eps, min_pts)
    "moons": (*datasets.make_moons(500, noise=0.07, random_state=0), 0.15, 5),
    "circles": (*datasets.make_circles(600, factor=0.5, noise=0.05, random_state=0), 0.15, 5),
    "smiley": (*smiley(), 0.3, 5),
    "blobs + 300 noise": (*noisy_blobs(), 0.55, 12),
}
fig, axes = plt.subplots(2, 4, figsize=(12, 5.6))
rows = []
for j, (name, (X, y, eps, mp)) in enumerate(wins.items()):
    km = ...
    db = ...
    rows.append({"dataset": name, "ARI k-means": ari(y, km), "ARI DBSCAN": ari(y, db), "DBSCAN clusters": n_clusters(db),
                 "DBSCAN noise": int(np.sum(db == -1)), "true noise": int(np.sum(y == -1))})
    plot_labels(axes[0, j], X, km, s=4, title=f"{name}: k-means ARI={ari(y, km):.2f}")
    plot_labels(axes[1, j], X, db, s=4, title=f"DBSCAN eps={eps}, mp={mp}: ARI={ari(y, db):.2f}")
fig.tight_layout(); plt.show()
pd.DataFrame(rows).round(3)

*Your answer here.*

**d)** Now a case where DBSCAN loses: two tight blobs close together and a much sparser third blob further away. Before running, predict whether any $(\varepsilon, \text{minPts})$ recovers all three clusters, and what DBSCAN does at a small and at a large $\varepsilon$. Why does k-means not have this problem?

In [ ]:
rng = np.random.default_rng(1)
X_vd = np.vstack([rng.normal([0, 0], 0.15, (200, 2)), rng.normal([1.0, 0], 0.15, (200, 2)),
                  rng.normal([3.5, 2.5], 0.8, (200, 2))])
y_vd = np.repeat([0, 1, 2], 200)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X_vd)
fig, axes = plt.subplots(1, 4, figsize=(13, 3), gridspec_kw={"width_ratios": [1, 1, 1, 1.4]})
plot_labels(axes[0], X_vd, km, s=4, title=f"k-means k=3: ARI={ari(y_vd, km):.2f}")
for ax, e in zip(axes[1:3], [0.1, 0.6]):
    lab = DBSCAN(eps=e, min_samples=10).fit_predict(X_vd)
    plot_labels(ax, X_vd, lab, s=4, title=f"eps={e}, mp=10: k={n_clusters(lab)}, noise={np.sum(lab == -1)}, ARI={ari(y_vd, lab):.2f}")
eps_r, best = np.round(np.arange(0.03, 1.2, 0.03), 2), (-1, None, None)
for mp in [3, 5, 10, 20, 30]:
    scores = [ari(y_vd, DBSCAN(eps=e, min_samples=mp).fit_predict(X_vd)) for e in eps_r]
    axes[3].plot(eps_r, scores, label=f"minPts={mp}")
    i = int(np.argmax(scores))
    if scores[i] > best[0]:
        best = (scores[i], eps_r[i], mp)
axes[3].axhline(ari(y_vd, km), color="k", ls="--", label="k-means")
axes[3].set_xlabel(r"$\varepsilon$"); axes[3].set_ylabel("ARI"); axes[3].legend(fontsize=6, ncol=2, loc="center right")
fig.tight_layout(); plt.show()
print(f"k-means ARI = {ari(y_vd, km):.3f} | best DBSCAN ARI over the grid = {best[0]:.3f} (eps={best[1]}, minPts={best[2]})")

*Your answer here.*

## 6. Soft clustering: Gaussian mixture models and EM

A Gaussian mixture model (GMM) gives every point a *probability* of belonging to each cluster instead of a single label. Here we look at what shapes those probabilities (priors, variances) and when a GMM beats k-means. Notation: component $S_j$ with prior $P(S_j)$, mean $\mu_j$, variance $\sigma_j^2$ (covariance $\Sigma_j$ in 2-D), and responsibility $r_{ij}=P(S_j\mid x_i)$.

> **Before this section:** complete Worksheet 8 (one round of EM on a 1-D Gaussian mixture).

In [ ]:
from scipy.stats import norm
from scipy.special import logsumexp
from matplotlib.patches import Ellipse
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score


def posterior(x, mus, sigmas, priors):
    """(n, k) array of P(S_j | x_i) by Bayes' rule. `sigmas` are STANDARD DEVIATIONS (scipy's `scale`)."""
    x = np.asarray(x, dtype=float)
    weighted = np.asarray(priors) * norm.pdf(x[:, None], loc=mus, scale=sigmas)   # P(S_j) P(x_i | S_j)
    return weighted / weighted.sum(axis=1, keepdims=True)


def find_boundaries(xs, post):
    """x-locations where a posterior curve crosses 0.5 (linear interpolation on the grid xs)."""
    idx = np.where(np.diff((post > 0.5).astype(int)) != 0)[0]
    return xs[idx] + (0.5 - post[idx]) * (xs[idx + 1] - xs[idx]) / (post[idx + 1] - post[idx])


def draw_ellipses(ax, mus, Sigmas, nstd=2):
    """Draw the nstd-standard-deviation contour of each 2-D Gaussian (mu_j, Sigma_j)."""
    for j, (mu, S) in enumerate(zip(mus, Sigmas)):
        vals, vecs = np.linalg.eigh(S)
        angle = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
        ax.add_patch(Ellipse(mu, 2 * nstd * np.sqrt(vals[-1]), 2 * nstd * np.sqrt(vals[0]), angle=angle,
                             fill=False, color=plt.cm.tab10(j), lw=1.5))
        ax.plot(*mu, "kx", ms=7, mew=2)

### 6.1 Bayes' rule: priors and variances shape the boundary

With known species parameters, $P(S_j\mid x) = \dfrac{P(S_j)\,\mathcal N(x\mid\mu_j,\sigma_j^2)}{\sum_l P(S_l)\,\mathcal N(x\mid\mu_l,\sigma_l^2)}$, computed by the provided `posterior`. **Pair A:** Stegosaurus weights $\sim\mathcal N(4.5,\,0.8^2)$ tons, Triceratops $\sim\mathcal N(7,\,0.8^2)$ tons; with equal priors the decision boundary (both posteriors $=0.5$) is at $5.75$ tons.

**a)** Before running the next cell, predict: as $P(\text{Stego})$ goes from $0.8$ to $0.95$ (up from the equal-prior baseline above), does the boundary move toward the Stegosaurus mean or the Triceratops mean? Does the *shape* of either species' density $P(x\mid S_j)$ change? Then check the printed boundaries.

*Your answer here.*

In [ ]:
xs = np.linspace(1, 11, 2001)
mus_A, sd_A = np.array([4.5, 7.0]), np.array([0.8, 0.8])
fig, axes = plt.subplots(2, 2, figsize=(8, 4.6), sharex=True)
for col, p_stego in enumerate([0.8, 0.95]):
    priors = np.array([p_stego, 1 - p_stego])
    weighted = priors * norm.pdf(xs[:, None], loc=mus_A, scale=sd_A)
    post = posterior(xs, mus_A, sd_A, priors)
    b = find_boundaries(xs, post[:, 0])
    for j, name in enumerate(["Stegosaurus", "Triceratops"]):
        axes[0, col].plot(xs, weighted[:, j], color=plt.cm.tab10(j), label=name)
        axes[1, col].plot(xs, post[:, j], color=plt.cm.tab10(j))
    for ax in axes[:, col]:
        ax.axvline(b[0], color="k", ls="--", lw=1)
    axes[0, col].set_title(f"P(Stego)={p_stego}: boundary {b[0]:.2f} t", fontsize=10)
    print(f"P(Stego) = {p_stego:4}: boundary at {b[0]:.3f} tons")
axes[0, 0].set_ylabel("P(S_j) P(x | S_j)"); axes[1, 0].set_ylabel("P(S_j | x)")
axes[0, 0].legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("weight (tons)")
plt.tight_layout(); plt.show()

**Pair B:** Stegosaurus $\sim\mathcal N(4.5,\,0.6^2)$ (consistent adults) and Iguanodon $\sim\mathcal N(6,\,2^2)$ (juveniles and giant adults), equal priors.

In [ ]:
mus_B, sd_B, pri_B = np.array([4.5, 6.0]), np.array([0.6, 2.0]), np.array([0.5, 0.5])
xs = np.linspace(-2, 14, 3201)
post_B = posterior(xs, mus_B, sd_B, pri_B)
b_B = find_boundaries(xs, post_B[:, 0])
print("Pair B boundaries (tons):", b_B.round(2))
print("P(Iguanodon | x) at x = 1, 4.5, 10 tons:", posterior([1, 4.5, 10], mus_B, sd_B, pri_B)[:, 1].round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for j, name in enumerate(["Stegosaurus (sd 0.6)", "Iguanodon (sd 2)"]):
    axes[0].plot(xs, pri_B[j] * norm.pdf(xs, mus_B[j], sd_B[j]), color=plt.cm.tab10(j), label=name)
    axes[1].plot(xs, post_B[:, j], color=plt.cm.tab10(j))
for ax in axes:
    for bb in b_B:
        ax.axvline(bb, color="k", ls="--", lw=1)
    ax.set_xlabel("weight (tons)")
axes[0].set_title("weighted densities"); axes[0].legend(fontsize=8)
axes[1].set_title(f"posteriors: boundaries at {b_B[0]:.2f} and {b_B[1]:.2f} t")
plt.tight_layout(); plt.show()

**b)** Pair B has **two** decision boundaries: very light *and* very heavy animals are both called Iguanodon. Using the formula for $\mathcal N(x\mid\mu,\sigma^2)$, explain why the wider species wins on both tails, however far out you go. Why can 1-D k-means with $k=2$ never produce this assignment, whatever its centroids?

*Your answer here.*

**c)** `scipy.stats.norm.pdf(x, loc, scale)` expects the **standard deviation** as `scale`, and it is an easy slip to pass the variance $\sigma^2$ instead. The next cell compares the two for Pair B and for a pair with both $\sigma=1$. What goes wrong for Pair B, and in which direction? Why does the mistake go completely unnoticed when every $\sigma_j=1$, and what does that suggest about choosing test cases?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
cases = [("Pair B (sd 0.6 and 2)", mus_B, sd_B), ("both sd = 1", np.array([4.5, 7.0]), np.array([1.0, 1.0]))]
for ax, (title, m, s) in zip(axes, cases):
    good = posterior(xs, m, s, pri_B)[:, 0]
    wrong = posterior(xs, m, s ** 2, pri_B)[:, 0]          # variance passed as `scale`
    ax.plot(xs, good, label="scale = sd (correct)")
    ax.plot(xs, wrong, "--", label="scale = variance")
    ax.set_title(f"{title}\nboundaries: sd {find_boundaries(xs, good).round(2)}, var {find_boundaries(xs, wrong).round(2)}",
                 fontsize=9)
    ax.set_xlabel("weight (tons)"); ax.set_ylabel("P(Stego | x)")
    print(f"{title:22s} max |P(sd) - P(var)| = {np.abs(good - wrong).max():.3f}, "
          f"peak P(Stego|x): {good.max():.2f} vs {wrong.max():.2f}")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

*Your answer here.*

### 6.2 The data decides: GMM vs k-means

We compare `KMeans` with `GaussianMixture(covariance_type="full")` on five labeled datasets: **anisotropic** (three tilted clusters), **core + halo** (a tight core, sd $0.5$, inside a wide halo, sd $3$, same center), **unequal proportions** ($950$ vs $50$ points, same spread), **moons**, and **equal spherical** blobs. ARI: $1$ = matches the true labels perfectly, about $0$ = no better than random.

**d)** Before running, predict for each dataset whether the GMM clearly wins, they tie, or both fail. Then fill in the two model lines and use the table: (i) which GMM parameter ($\mu_j$, $\Sigma_j$ or $P(S_j)$) is responsible for each GMM win? (ii) Why do both fail on the moons? (iii) In what sense is k-means a special case of a GMM? The second cell runs EM with one fixed, shared variance $\sigma^2 I$ and equal priors (only the means are learned) and compares it with Lloyd's algorithm from the same start as $\sigma^2$ shrinks.

In [ ]:
X_an, y_an = datasets.make_blobs(n_samples=600, centers=3, random_state=170)
X_an = X_an @ np.array([[0.6, -0.64], [-0.41, 0.85]])                      # shear: tilted clusters
rng = np.random.default_rng(4)
X_ch = np.vstack([rng.normal(0, 0.5, size=(300, 2)), rng.normal(0, 3.0, size=(300, 2))])
rng = np.random.default_rng(2)
X_up = np.vstack([rng.normal([0, 0], 1.0, size=(950, 2)), rng.normal([4.5, 0], 1.0, size=(50, 2))])
X_mo, y_mo = datasets.make_moons(n_samples=400, noise=0.07, random_state=0)
X_sp, y_sp = datasets.make_blobs(n_samples=600, centers=[[0, 0], [5, 0], [2.5, 4.3]], cluster_std=1.3, random_state=0)
scenarios = [("anisotropic", X_an, y_an, 3), ("core + halo", X_ch, np.repeat([0, 1], 300), 2),
             ("unequal proportions", X_up, np.repeat([0, 1], [950, 50]), 2), ("moons", X_mo, y_mo, 2),
             ("equal spherical", X_sp, y_sp, 3)]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
print(f"{'dataset':20s} {'k-means ARI':>11s} {'GMM ARI':>8s}   GMM weights")
for col, (name, X, y, k) in enumerate(scenarios):
    km_lab = ...
    gm = ...
    gm_lab = gm.predict(X)
    a_km, a_gm = adjusted_rand_score(y, km_lab), adjusted_rand_score(y, gm_lab)
    print(f"{name:20s} {a_km:11.2f} {a_gm:8.2f}   {gm.weights_.round(2)}")
    for row, (lab, title, a) in enumerate([(km_lab, "k-means", a_km), (gm_lab, "GMM", a_gm)]):
        ax = axes[row, col]
        ax.scatter(*X.T, c=lab, cmap="tab10", vmin=0, vmax=9, s=5, alpha=0.6)
        if row == 1:
            draw_ellipses(ax, gm.means_, gm.covariances_)
        ax.set_title(f"{name}\n{title}: ARI = {a:.2f}", fontsize=9)
        ax.set_aspect("equal", adjustable="datalim"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# k-means as a limit of EM: shared fixed variance s2 * I, equal priors, only the means are updated
start = X_an[np.random.default_rng(0).choice(len(X_an), 3, replace=False)]
km_an = KMeans(n_clusters=3, init=start, n_init=1, tol=0).fit(X_an)          # Lloyd's from the same start
print(f"{'sigma^2':>8s} {'mean max r_ij':>14s} {'ARI vs Lloyd':>13s} {'max |mu - Lloyd centroid|':>26s}")
for s2 in [5.0, 1.0, 0.1, 1e-3]:
    mus = start.copy()
    for _ in range(300):
        A = -((X_an[:, None, :] - mus[None]) ** 2).sum(axis=2) / (2 * s2)    # log N(x_i | mu_j, s2 I) + const
        R = np.exp(A - logsumexp(A, axis=1, keepdims=True))
        mus = R.T @ X_an / R.sum(axis=0)[:, None]
    gap = max(np.linalg.norm(km_an.cluster_centers_ - m, axis=1).min() for m in mus)
    print(f"{s2:8g} {R.max(axis=1).mean():14.3f} {adjusted_rand_score(km_an.labels_, R.argmax(axis=1)):13.3f} {gap:26.4f}")

*Your answer here.*

## 7. Clustering aggregation: comparing clusterings

Here clusterings themselves become the data. We compare two clusterings with the disagreement distance $D$.

In [ ]:
from sklearn.metrics import rand_score, adjusted_rand_score


def n_pairs(n):
    """Number of unordered pairs among n points: n(n-1)/2."""
    return n * (n - 1) // 2

### 7.1 The disagreement distance

For clusterings $P$ and $C$ of the same $n$ points, let $I_{P,C}(x,y) = 1$ if $P$ and $C$ **disagree** on the pair $\{x,y\}$ (together in one, apart in the other) and $0$ otherwise. Then $D(P,C) = \sum_{\{x,y\}} I_{P,C}(x,y)$, a sum over all $\binom n2$ unordered pairs. $D$ only asks "together or apart?", so label names never need to match. The **Rand index** $\mathrm{RI}$ (sklearn's `rand_score`) is the fraction of pairs on which two clusterings agree, so $D = (1-\mathrm{RI})\binom n2$.

**a)** (i) Implement `disagreement_distance(P, C)` directly from the definition by looping over all pairs. (ii) Complete the one-line `disagreement_from_rand` using the Rand index; the check cell compares the two. (iii) In the table below, two *unrelated* random clusterings with 10 clusters have a Rand index of about $0.8$. Why so high, and what does that say about reading raw values of $D$? (ARI: 1 = matches the other clustering perfectly, about 0 = no better than random.)

In [ ]:
def disagreement_distance(P, C):
    """Number of unordered pairs {x, y} on which clusterings P and C disagree
    (together in one clustering, apart in the other). Labels can be ints, strings, ..."""
    P, C = list(P), list(C)
    # YOUR CODE HERE
    ...


def disagreement_from_rand(P, C):
    """D(P, C) computed from sklearn's Rand index (fast; used in the rest of the section)."""
    return ...

In [ ]:
# Check your work
assert disagreement_distance([1, 1, 2, 3, 3], list("abacd")) == 3
rng = np.random.default_rng(0)
for trial in range(50):
    n = int(rng.integers(2, 60))
    P, C = rng.integers(0, rng.integers(1, 6), n), rng.integers(0, rng.integers(1, 6), n)
    assert disagreement_distance(P, C) == disagreement_from_rand(P, C)
print("disagreement_distance matches (1 - RI) * n(n-1)/2\n")

n = 1000
P = rng.integers(0, 10, n)
cases = {"P vs P with labels renamed": rng.permutation(10)[P],
         "P vs independent random, 10 clusters": rng.integers(0, 10, n),
         "P vs all singletons": np.arange(n)}
rows = [[name, disagreement_from_rand(P, Q), disagreement_from_rand(P, Q) / n_pairs(n),
         rand_score(P, Q), adjusted_rand_score(P, Q)] for name, Q in cases.items()]
print(pd.DataFrame(rows, columns=["comparison", "D", "D / #pairs", "Rand index", "ARI"]).round(3).to_string(index=False))

*Your answer here.*

**b)** The price of a mistake. The truth has clusters of sizes $400$, $100$ and $20$, and we make one mistake at a time. Before running, rank these from cheapest to most expensive in $D$: moving **one** point from the 400-cluster to the 20-cluster; **shattering** the 20-cluster into singletons; splitting the 20-cluster in half; merging the 100- and 20-clusters; splitting the 400-cluster in half; merging the 400- and 100-clusters. Then explain which kinds of mistakes $D$ punishes hardest, which it barely notices, and when that would mislead you.

In [ ]:
sizes = [400, 100, 20]
truth = np.repeat([0, 1, 2], sizes)

def split(lab, c):           # split cluster c into two halves
    new = lab.copy(); idx = np.where(lab == c)[0]; new[idx[: len(idx) // 2]] = lab.max() + 1; return new
def merge(lab, c1, c2):      # merge cluster c2 into c1
    new = lab.copy(); new[lab == c2] = c1; return new
def move_one(lab, src, dst): # move a single point from cluster src to cluster dst
    new = lab.copy(); new[np.where(lab == src)[0][0]] = dst; return new
def shatter(lab, c):         # every point of cluster c becomes a singleton
    new = lab.copy(); idx = np.where(lab == c)[0]; new[idx] = lab.max() + 1 + np.arange(len(idx)); return new

mistakes = {"move 1 point 400 → 20": move_one(truth, 0, 2), "shatter the 20 into singletons": shatter(truth, 2),
            "split 20 → 10+10": split(truth, 2), "merge 100 + 20": merge(truth, 1, 2),
            "split 400 → 200+200": split(truth, 0), "merge 400 + 100": merge(truth, 0, 1)}
rows = [[name, disagreement_from_rand(truth, lab), adjusted_rand_score(truth, lab)] for name, lab in mistakes.items()]
print(pd.DataFrame(rows, columns=["mistake", "D", "ARI"]).sort_values("D").round(4).to_string(index=False))

*Your answer here.*

### 7.2 Is $D$ a metric?

A metric needs $D(C,P)=0$ iff $C=P$, symmetry, and the triangle inequality $D(C_1,C_3) \le D(C_1,C_2) + D(C_2,C_3)$. The cell below tries to break it with thousands of random triples of clusterings of 10 points.

In [ ]:
rng = np.random.default_rng(4)
D = disagreement_distance
slacks, sym_fail = [], 0
for _ in range(3000):
    C1, C2, C3 = (rng.integers(0, rng.integers(1, 5), 10) for _ in range(3))
    slacks.append(D(C1, C2) + D(C2, C3) - D(C1, C3))
    sym_fail += D(C1, C2) != D(C2, C1)
slacks = np.array(slacks)
print(f"triangle-inequality violations: {np.sum(slacks < 0)} / 3000, smallest slack {slacks.min()}, "
      f"symmetry failures: {sym_fail}")
a, b = np.array([0, 0, 1, 1, 2, 2]), np.array([5, 5, 0, 0, 3, 3])
print(f"D({a}, {b}) = {D(a, b)}, although the label vectors differ")

**c)** No violations. Prove the triangle inequality one pair at a time: for a fixed pair $\{x,y\}$ show $I_{C_1,C_3}(x,y) \le I_{C_1,C_2}(x,y) + I_{C_2,C_3}(x,y)$, then explain why summing over pairs finishes the proof. Given the last printed line, is $D$ a metric on label vectors or on partitions?

*Your answer here.*

## 8. Which tool for which data?

Every clustering algorithm assumes something about what a cluster is: a compact ball around a mean (k-means), a Gaussian with its own shape (GMM), a chain of nearby points (single link), a balanced group of similar diameter (complete link, average link), a low-variance group (Ward), or a dense region surrounded by sparse space (DBSCAN). This section runs all of them on the same data, then gives you a real dataset where you make the choices yourself.

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score


def noise_to_singletons(labels):
    """Relabel every noise point (label -1) as its own one-point cluster.

    Used for BOTH the true labels and the predicted labels, so that
    (i) calling a real cluster "noise" costs score, and
    (ii) absorbing true background noise into a cluster also costs score.
    """
    labels = np.asarray(labels).copy()
    noise = labels < 0
    labels[noise] = labels.max() + 1 + np.arange(noise.sum())
    return labels


def score(y_true, y_pred):
    """Adjusted Rand index (1 = perfect agreement with the truth, about 0 = random)."""
    return adjusted_rand_score(noise_to_singletons(y_true), noise_to_singletons(y_pred))

### 8.1 Seven algorithms, eight datasets

Each 2-D dataset below stresses a different assumption. Points labelled $-1$ are background noise that belongs to no cluster. Scores are the adjusted Rand index (ARI: $1$ = matches the true labels perfectly, about $0$ = no better than random), with noise points counted as one-point clusters.

In [ ]:
def make_gallery(seed=0):
    rng = np.random.default_rng(seed)
    D = {}
    X, y = datasets.make_blobs(400, centers=[[-4, 0], [0, 3], [4, 0]], cluster_std=0.9, random_state=1)
    D["round blobs"] = (X, y)
    X, y = datasets.make_blobs(400, centers=3, cluster_std=0.6, random_state=170)
    D["stretched blobs"] = (X @ np.array([[0.6, -0.64], [-0.41, 0.85]]), y)
    D["moons"] = datasets.make_moons(400, noise=0.06, random_state=0)
    D["rings"] = datasets.make_circles(400, factor=0.45, noise=0.05, random_state=0)
    Xb, yb = datasets.make_blobs(300, centers=[[-2, -2], [2, 2], [2, -2]], cluster_std=0.45, random_state=3)
    Xn = rng.uniform(-4.5, 4.5, (100, 2))
    D["blobs + noise"] = (np.vstack([Xb, Xn]), np.r_[yb, -np.ones(100, int)])
    X1, X2 = rng.normal([0, 0], 0.25, (200, 2)), rng.normal([3.2, 0], 1.3, (200, 2))
    D["dense + sparse"] = (np.vstack([X1, X2]), np.repeat([0, 1], 200))
    X1, X2 = rng.normal([-3, 0], 0.6, (180, 2)), rng.normal([3, 0], 0.6, (180, 2))
    t = np.linspace(-2.2, 2.2, 40)
    bridge = np.c_[t, rng.normal(0, 0.05, 40)]
    D["bridged blobs"] = (np.vstack([X1, X2, bridge]), np.r_[np.zeros(180, int), np.ones(180, int), (t > 0).astype(int)])
    X1, X2 = rng.normal([0, 0], 1.0, (450, 2)), rng.normal([3.3, 0], 0.35, (40, 2))
    D["big + small"] = (np.vstack([X1, X2]), np.r_[np.zeros(450, int), np.ones(40, int)])
    return D


gallery = make_gallery()
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for ax, (name, (X, y)) in zip(axes.ravel(), gallery.items()):
    ax.scatter(X[:, 0], X[:, 1], c=np.where(y < 0, -1, y), cmap="tab10", s=6, vmin=-1, vmax=9)
    ax.set_title(name); ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal", "datalim")
plt.suptitle("Ground truth (noise shown in the first colour)"); plt.tight_layout(); plt.show()

**a)** Before running anything, fill in the table: for each dataset, which of **k-means**, **GMM** (full covariance), **single link**, **complete link**, **average link**, **Ward**, and **DBSCAN** do you expect to do well (✓), partially (~), or badly (✗)?

| dataset | k-means | GMM | single link | complete link | average link | Ward | DBSCAN |
|---|---|---|---|---|---|---|---|
| round blobs | | | | | | | |
| stretched blobs | | | | | | | |
| moons | | | | | | | |
| rings | | | | | | | |
| blobs + noise | | | | | | | |
| dense + sparse | | | | | | | |
| bridged blobs | | | | | | | |
| big + small | | | | | | | |

*Your answer here.*

Every algorithm gets its best shot: the true $k$ for k-means, GMM and the four linkages, and for DBSCAN the best $(\varepsilon, \text{minPts})$ from a small grid, **chosen by peeking at the true labels**.

**b)** Complete `run_all` (use `n_init=10, random_state=0` for k-means and `covariance_type="full", n_init=5, random_state=0` for the GMM) and run the next two cells. Then compare the scores with your predictions and explain: (i) why single link scores about $0$ on the *easiest* dataset (round blobs) but perfectly on rings and big + small; (ii) why k-means and Ward score only about $0.6$ on blobs + noise even though the three blobs themselves are found; (iii) why GMM beats k-means by a wide margin on dense + sparse.

In [ ]:
def best_dbscan(X, y, eps_grid=np.linspace(0.05, 1.5, 30), min_pts_grid=(5, 10)):
    """Oracle-tuned DBSCAN: returns (labels, eps, min_pts) with the best score against y."""
    best = (-np.inf, None, None, None)
    for eps in eps_grid:
        for mp in min_pts_grid:
            lab = DBSCAN(eps=eps, min_samples=mp).fit_predict(X)
            s = score(y, lab)
            if s > best[0]:
                best = (s, lab, float(eps), mp)
    return best[1], best[2], best[3]


def run_all(X, y):
    """Return a dict {algorithm name: predicted labels} for the seven algorithms."""
    k = len(set(y[y >= 0]))
    out = {}
    out["k-means"] = ...
    out["GMM"] = ...
    out["single link"] = ...
    out["complete link"] = ...
    out["average link"] = ...
    out["Ward"] = ...
    out["DBSCAN"], eps, mp = best_dbscan(X, y)
    return out


t0 = time.time()
results = {name: run_all(X, y) for name, (X, y) in gallery.items()}
scores = pd.DataFrame({name: {alg: score(gallery[name][1], lab) for alg, lab in res.items()}
                       for name, res in results.items()}).T.round(2)
print(f"ran 7 algorithms on 8 datasets in {time.time() - t0:.1f}s")
scores

In [ ]:
algs = list(next(iter(results.values())).keys())
fig, axes = plt.subplots(len(gallery), len(algs), figsize=(14, 15))
for i, (name, (X, y)) in enumerate(gallery.items()):
    for j, alg in enumerate(algs):
        lab = results[name][alg]
        ax = axes[i, j]
        ax.scatter(X[lab >= 0, 0], X[lab >= 0, 1], c=lab[lab >= 0] % 10, cmap="tab10", s=3, vmin=0, vmax=9)
        ax.scatter(X[lab < 0, 0], X[lab < 0, 1], c="lightgray", s=3)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{alg}: {scores.loc[name, alg]:.2f}", fontsize=7)
        if j == 0:
            ax.set_ylabel(name, fontsize=8)
plt.tight_layout(); plt.show()

*Your answer here.*

**c)** DBSCAN was tuned by looking at the true labels. Pick two datasets and describe how you would choose $\varepsilon$ and $\text{minPts}$ **without** labels. For which dataset in the gallery would your label-free choice be most likely to disagree with the oracle choice, and why?

*Your answer here.*